[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fvalenzuelag/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/main/RA1/IL1.2/3-chain-of-thought.ipynb)


# 3. Chain-of-Thought (CoT) Prompting - Razonamiento Paso a Paso

## Objetivos de Aprendizaje
- Comprender el concepto de Chain-of-Thought (CoT) prompting
- Implementar CoT con y sin ejemplos
- Aplicar CoT a problemas complejos y razonamiento lógico
- Combinar CoT con otras técnicas de prompting

## ¿Qué es Chain-of-Thought?

Chain-of-Thought (CoT) es una técnica que hace que el modelo "**piense en voz alta**" mostrando su proceso de razonamiento paso a paso antes de llegar a la respuesta final.

### Principio Básico:
En lugar de saltar directamente a la respuesta, el modelo:
1. **Descompone** el problema en pasos
2. **Razona** cada paso explícitamente  
3. **Construye** hacia la solución final
4. **Proporciona** la respuesta con justificación

### Ventajas:
- **Mejor precisión**: Especialmente en problemas complejos
- **Transparencia**: Puedes ver el razonamiento
- **Debugging**: Identificar dónde falla el razonamiento
- **Confianza**: Mayor seguridad en la respuesta

### Casos de Uso Ideales:
- Problemas matemáticos
- Razonamiento lógico
- Análisis complejos
- Toma de decisiones
- Resolución de problemas multi-paso

In [1]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain langchain-classic langchain-openai python-dotenv


In [2]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
    os.environ.setdefault("LLM_MODEL", "llama-3.3-70b-versatile")
    os.environ.setdefault("LLM_MODEL_SMALL", "llama-3.1-8b-instant")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

# Configuración inicial
from langchain_openai import ChatOpenAI
# LangChain v1: langchain_core.messages / langchain_core.documents
from langchain_classic.schema import HumanMessage
import os
import time

# Configurar el modelo
llm = ChatOpenAI(
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
    model=os.getenv("LLM_MODEL", "llama-3.3-70b-versatile"),
    temperature=0.1  # Baja temperatura para razonamiento más consistente
)

print("✓ Modelo configurado para Chain-of-Thought")
print("✓ Temperature baja para razonamiento consistente")

✓ Modelo configurado para Chain-of-Thought
✓ Temperature baja para razonamiento consistente


## Comparación: Sin CoT vs Con CoT

Veamos la diferencia dramática que puede hacer CoT en problemas complejos.

In [3]:
# Comparación directa: razonamiento directo vs CoT
def comparar_sin_vs_con_cot():
    print("=== COMPARACIÓN: SIN CoT vs CON CoT ===")
    
    # Problema complejo que requiere múltiples pasos
    problema = """Una tienda tiene una promoción: 'Compra 2 productos y obtén 30% de descuento en el más barato'. 
    Juan compra una camiseta de 45€, unos zapatos de 120€ y una chaqueta de 80€. 
    ¿Cuánto paga en total?"""
    
    # Prompt sin CoT
    prompt_sin_cot = f"""Resuelve este problema:
    
{problema}
    
Respuesta:"""
    
    # Prompt con CoT
    prompt_con_cot = f"""Resuelve este problema paso a paso:
    
{problema}
    
Piensa paso a paso:
1. Primero identifica los productos y precios
2. Determina cómo se aplica la promoción
3. Calcula el descuento
4. Calcula el total final
    
Razonamiento:"""
    
    # Probar sin CoT
    print("\n1. SIN CHAIN-OF-THOUGHT:")
    print("-" * 30)
    try:
        response_sin = llm.invoke([HumanMessage(content=prompt_sin_cot)])
        print(response_sin.content)
    except Exception as e:
        print(f"Error: {e}")
    
    print("\n" + "="*60)
    
    # Probar con CoT
    print("\n2. CON CHAIN-OF-THOUGHT:")
    print("-" * 30)
    try:
        response_con = llm.invoke([HumanMessage(content=prompt_con_cot)])
        print(response_con.content)
    except Exception as e:
        print(f"Error: {e}")
    
    print("\n=== ANÁLISIS ===")
    print("• Sin CoT: Puede llegar a respuesta incorrecta o saltar pasos")
    print("• Con CoT: Muestra razonamiento completo y reduce errores")
    print("• CoT especialmente útil para problemas multi-paso")

# Ejecutar comparación
comparar_sin_vs_con_cot()

=== COMPARACIÓN: SIN CoT vs CON CoT ===

1. SIN CHAIN-OF-THOUGHT:
------------------------------


Para resolver este problema, debemos seguir los pasos de la promoción:

1. **Identificar los productos**: Juan compra 3 productos: una camiseta de 45€, unos zapatos de 120€ y una chaqueta de 80€.

2. **Aplicar la promoción**: La promoción es "Compra 2 productos y obtén 30% de descuento en el más barato". Dado que Juan compra 3 productos, podemos considerar que está comprando dos productos y obteniendo el descuento en el más barato de los dos más baratos, o simplemente aplicar el descuento en el producto más barato de los tres.

3. **Ordenar los productos por precio**: 
   - Camiseta: 45€
   - Chaqueta: 80€
   - Zapatos: 120€

4. **Aplicar el descuento**: El producto más barato es la camiseta de 45€. El 30% de 45€ es 0.30 * 45 = 13.50€. Entonces, el descuento en la camiseta es de 13.50€.

5. **Calcular el precio final de la camiseta**: 45€ - 13.50€ = 31.50€.

6. **Calcular el total**: 
   - Camiseta con descuento: 31.50€
   - Chaqueta: 80€
   - Zapatos: 120€
   - Total: 31.50€ + 80€ + 1

Para resolver este problema, seguiremos los pasos que has indicado:

**1. Identifica los productos y precios:**
   - Camiseta: 45€
   - Zapatos: 120€
   - Chaqueta: 80€

**2. Determina cómo se aplica la promoción:**
   - La promoción es "Compra 2 productos y obtén 30% de descuento en el más barato". Esto significa que, de los tres productos, el más barato recibirá un descuento del 30% después de comprar al menos dos productos.

**3. Calcula el descuento:**
   - Primero, debemos identificar el producto más barato, que es la camiseta con un precio de 45€.
   - El descuento es del 30% sobre el precio de la camiseta.
   - Descuento = 30% de 45€ = 0.30 * 45 = 13.50€

**4. Calcula el total final:**
   - El precio de la camiseta después del descuento = 45€ - 13.50€ = 31.50€
   - El precio total de los productos después del descuento = Precio de la camiseta con descuento + Precio de los zapatos + Precio de la chaqueta
   - Total = 31.50€ + 120€ + 80€ = 231.50€

Por lo tanto, Juan pagará un tot

## Zero-Shot Chain-of-Thought

La forma más simple de CoT: simplemente pedirle al modelo que "piense paso a paso".

In [4]:
# Zero-shot CoT: solo agregar "piensa paso a paso"
def zero_shot_cot():
    print("=== ZERO-SHOT CHAIN-OF-THOUGHT ===")
    
    problemas = [
        "Si un tren viaja a 80 km/h y necesita llegar a una ciudad que está a 240 km, pero se detiene 15 minutos en una estación intermedia, ¿cuánto tiempo total toma el viaje?",
        "Una empresa tiene 150 empleados. El 40% trabaja en desarrollo, el 25% en ventas, el 20% en marketing y el resto en administración. Si cada empleado de desarrollo gana 50,000€ anuales, ¿cuál es el costo anual solo del departamento de desarrollo?",
        "María tiene el triple de edad que su hermana Ana. En 5 años, María tendrá el doble de la edad que tendrá Ana. ¿Cuántos años tiene cada una ahora?"
    ]
    
    for i, problema in enumerate(problemas, 1):
        print(f"\n{i}. PROBLEMA:")
        print(f"{problema}")
        
        # Prompt zero-shot CoT
        prompt = f"{problema}\n\nPiensa paso a paso:"
        
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            print("\nSOLUCIÓN:")
            print(response.content)
            
            # Análisis básico del razonamiento
            pasos = response.content.count('\n')
            tiene_calculo = any(op in response.content for op in ['+', '-', '*', '/', '=', '%'])
            
            print(f"\nAnálisis: {pasos} líneas de razonamiento, {'con' if tiene_calculo else 'sin'} cálculos explícitos")
            
        except Exception as e:
            print(f"Error: {e}")
        
        print("-" * 80)

# Ejecutar zero-shot CoT
zero_shot_cot()

=== ZERO-SHOT CHAIN-OF-THOUGHT ===

1. PROBLEMA:
Si un tren viaja a 80 km/h y necesita llegar a una ciudad que está a 240 km, pero se detiene 15 minutos en una estación intermedia, ¿cuánto tiempo total toma el viaje?



SOLUCIÓN:
Para calcular el tiempo total del viaje, debemos considerar tanto el tiempo que el tren pasa en movimiento como el tiempo que pasa detenido en la estación intermedia.

1. **Calcular el tiempo de viaje sin detenciones**: 
   - La distancia total a recorrer es de 240 km.
   - La velocidad del tren es de 80 km/h.
   - El tiempo de viaje sin detenciones se calcula dividiendo la distancia por la velocidad: 
     - Tiempo = Distancia / Velocidad
     - Tiempo = 240 km / 80 km/h = 3 horas

2. **Calcular el tiempo de detención**:
   - El tren se detiene durante 15 minutos en una estación intermedia.
   - Para sumar este tiempo al tiempo de viaje, debemos convertirlo a la misma unidad que el tiempo de viaje, que es horas. 
     - 15 minutos = 15/60 horas = 0.25 horas

3. **Calcular el tiempo total del viaje**:
   - Sumamos el tiempo de viaje sin detenciones y el tiempo de detención:
     - Tiempo total = Tiempo de viaje + Tiempo de detención
     - Tiempo total = 3 horas + 0.25 horas


SOLUCIÓN:
Para calcular el costo anual del departamento de desarrollo, necesitamos seguir los siguientes pasos:

1. **Calcular el número de empleados en desarrollo**: 
   - Total de empleados: 150
   - Porcentaje de empleados en desarrollo: 40%
   - Número de empleados en desarrollo = Total de empleados * Porcentaje de empleados en desarrollo
   - Número de empleados en desarrollo = 150 * 0.40 = 60 empleados

2. **Calcular el costo anual del departamento de desarrollo**:
   - Salario anual por empleado de desarrollo: 50,000€
   - Número de empleados en desarrollo: 60
   - Costo anual del departamento de desarrollo = Salario anual por empleado * Número de empleados en desarrollo
   - Costo anual del departamento de desarrollo = 50,000 * 60 = 3,000,000€

Por lo tanto, el costo anual solo del departamento de desarrollo es de **3,000,000€**.

Análisis: 14 líneas de razonamiento, con cálculos explícitos
--------------------------------------------------------------------------------

3. PR


SOLUCIÓN:
Analicemos el problema paso a paso:

1. **Definamos las variables**:
   - La edad actual de María sea M.
   - La edad actual de Ana sea A.

2. **Información dada**:
   - María tiene el triple de edad que su hermana Ana, por lo que podemos escribir la ecuación: M = 3A.
   - En 5 años, María tendrá el doble de la edad que tendrá Ana. Entonces, en 5 años, la edad de María será M + 5 y la edad de Ana será A + 5. Según el problema, M + 5 = 2(A + 5).

3. **Resolvamos las ecuaciones**:
   - Ya tenemos M = 3A.
   - Sustituyamos M en la segunda ecuación: 3A + 5 = 2(A + 5).
   - Simplifiquemos la ecuación: 3A + 5 = 2A + 10.
   - Restemos 2A de ambos lados: A + 5 = 10.
   - Restemos 5 de ambos lados: A = 5.

4. **Encontremos la edad de María**:
   - Ya sabemos que M = 3A y A = 5, entonces M = 3 * 5 = 15.

5. **Conclusión**:
   - Ana tiene 5 años ahora.
   - María tiene 15 años ahora.

Por lo tanto, Ana tiene 5 años y María tiene 15 años en este momento.

Análisis: 24 líneas de razonami

## Few-Shot Chain-of-Thought

Combinamos CoT con ejemplos para mostrar el patrón de razonamiento deseado.

In [5]:
# Few-shot CoT: ejemplos con razonamiento paso a paso
def few_shot_cot():
    print("=== FEW-SHOT CHAIN-OF-THOUGHT ===")
    
    # Nuevo problema para resolver
    nuevo_problema = "Un parking cobra 3€ la primera hora y 2€ cada hora adicional. Si alguien paga 15€, ¿cuántas horas estuvo estacionado?"
    
    # Prompt con ejemplos de razonamiento
    prompt_few_shot_cot = f"""Resuelve problemas matemáticos mostrando el razonamiento paso a paso:
    
Problema: Una pizza cuesta 12€ y cada ingrediente extra cuesta 1.50€. Si Pedro paga 18€, ¿cuántos ingredientes extra pidió?
Razonamiento:
1. Precio base de la pizza: 12€
2. Total pagado: 18€
3. Dinero gastado en extras: 18€ - 12€ = 6€
4. Costo por ingrediente extra: 1.50€
5. Número de ingredientes: 6€ ÷ 1.50€ = 4 ingredientes
Respuesta: Pedro pidió 4 ingredientes extra.
    
Problema: En una clase hay 24 estudiantes. Si se forman grupos de 6 estudiantes cada uno, ¿cuántos grupos se pueden formar? Si sobra algún estudiante, ¿cuántos?
Razonamiento:
1. Total de estudiantes: 24
2. Estudiantes por grupo: 6
3. División: 24 ÷ 6 = 4 grupos exactos
4. Verificación: 4 × 6 = 24 estudiantes
5. Resto: 24 - 24 = 0 estudiantes sobran
Respuesta: Se pueden formar 4 grupos completos y no sobra ningún estudiante.
    
Problema: {nuevo_problema}
Razonamiento:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_few_shot_cot)])
        print("PROBLEMA A RESOLVER:")
        print(nuevo_problema)
        print("\nSOLUCIÓN CON RAZONAMIENTO:")
        print(response.content)
        
        # Verificar si siguió el patrón
        razonamiento = response.content
        tiene_pasos_numerados = bool(re.search(r'\d+\.', razonamiento))
        tiene_calculos = any(op in razonamiento for op in ['=', '+', '-', '*', '/', '€'])
        tiene_respuesta_final = 'respuesta' in razonamiento.lower()
        
        print("\n=== ANÁLISIS DEL PATRÓN ===")
        print(f"✓ Pasos numerados: {'Sí' if tiene_pasos_numerados else 'No'}")
        print(f"✓ Cálculos explícitos: {'Sí' if tiene_calculos else 'No'}")
        print(f"✓ Respuesta final clara: {'Sí' if tiene_respuesta_final else 'No'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar few-shot CoT
import re
few_shot_cot()

=== FEW-SHOT CHAIN-OF-THOUGHT ===


PROBLEMA A RESOLVER:
Un parking cobra 3€ la primera hora y 2€ cada hora adicional. Si alguien paga 15€, ¿cuántas horas estuvo estacionado?

SOLUCIÓN CON RAZONAMIENTO:
Para resolver este problema, podemos seguir los siguientes pasos:

1. **Costo de la primera hora**: 3€
2. **Total pagado**: 15€
3. **Dinero gastado en horas adicionales**: 15€ - 3€ = 12€
4. **Costo por hora adicional**: 2€
5. **Número de horas adicionales**: 12€ ÷ 2€ = 6 horas adicionales
6. **Total de horas**: 1 hora (la primera) + 6 horas adicionales = 7 horas

**Respuesta**: La persona estuvo estacionada durante 7 horas.

=== ANÁLISIS DEL PATRÓN ===
✓ Pasos numerados: Sí
✓ Cálculos explícitos: Sí
✓ Respuesta final clara: Sí


## CoT para Razonamiento Lógico

Chain-of-Thought es especialmente poderoso para problemas de lógica y deducción.

In [6]:
# CoT para problemas de lógica
def cot_razonamiento_logico():
    print("=== CoT PARA RAZONAMIENTO LÓGICO ===")
    
    # Problema de lógica clásico
    problema_logica = """En una mesa redonda se sientan 5 personas: Ana, Bruno, Carlos, Diana y Elena.
    - Ana no está al lado de Bruno
    - Carlos está exactamente frente a Diana
    - Elena está al lado derecho de Ana
    - Bruno está al lado de Carlos
    
¿Cuál es la disposición completa alrededor de la mesa?"""
    
    prompt_logica = f"""Resuelve este problema de lógica paso a paso:
    
{problema_logica}
    
Razona sistemáticamente:
1. Identifica las restricciones
2. Establece relaciones conocidas
3. Deduce posiciones paso a paso
4. Verifica que se cumplan todas las condiciones
    
Proceso de deducción:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_logica)])
        print("PROBLEMA DE LÓGICA:")
        print(problema_logica)
        print("\nPROCESO DE RAZONAMIENTO:")
        print(response.content)
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar razonamiento lógico
cot_razonamiento_logico()

=== CoT PARA RAZONAMIENTO LÓGICO ===


PROBLEMA DE LÓGICA:
En una mesa redonda se sientan 5 personas: Ana, Bruno, Carlos, Diana y Elena.
    - Ana no está al lado de Bruno
    - Carlos está exactamente frente a Diana
    - Elena está al lado derecho de Ana
    - Bruno está al lado de Carlos

¿Cuál es la disposición completa alrededor de la mesa?

PROCESO DE RAZONAMIENTO:
**Paso 1: Identifica las restricciones**

1. Ana no está al lado de Bruno.
2. Carlos está exactamente frente a Diana.
3. Elena está al lado derecho de Ana.
4. Bruno está al lado de Carlos.

**Paso 2: Establece relaciones conocidas**

- Sabemos que Carlos y Diana están frente a frente, lo que significa que hay dos posiciones fijas en la mesa, aunque no sabemos exactamente dónde están en relación con los demás.
- Elena está al lado derecho de Ana, lo que nos da una relación de vecindad entre ellas.
- Bruno está al lado de Carlos, pero no sabemos si es a la izquierda o a la derecha.

**Paso 3: Deduce posiciones paso a paso**

1. **Carlos y Diana**: Como están 

In [7]:
# CoT para análisis de casos complejos
def cot_analisis_complejo():
    print("=== CoT PARA ANÁLISIS COMPLEJO ===")
    
    # Caso de negocio complejo
    caso_negocio = """Una startup de software tiene las siguientes métricas:
    - 10,000 usuarios activos mensuales
    - Tasa de conversión a premium: 5%
    - Precio premium: 29€/mes
    - Costo de adquisición por usuario: 15€
    - Retención mensual: 85%
    - Costos operativos mensuales: 12,000€
    
La empresa está considerando reducir el precio a 19€/mes para aumentar la conversión a 8%. 
¿Es una buena decisión financiera?"""
    
    prompt_analisis = f"""Analiza este caso de negocio paso a paso:
    
{caso_negocio}
    
Estructura tu análisis:
1. Calcula métricas del escenario actual
2. Calcula métricas del escenario propuesto
3. Compara ingresos y costos
4. Considera factores adicionales
5. Proporciona recomendación fundamentada
    
Análisis detallado:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_analisis)])
        print("CASO DE NEGOCIO:")
        print(caso_negocio)
        print("\nANÁLISIS PASO A PASO:")
        print(response.content)
        
        # Verificar completitud del análisis
        analisis = response.content.lower()
        elementos_clave = [
            'ingresos', 'costos', 'beneficio', 'actual', 'propuesto', 
            'recomendación', 'conversión', 'usuarios'
        ]
        elementos_presentes = sum(1 for elemento in elementos_clave if elemento in analisis)
        
        print(f"\n=== COMPLETITUD DEL ANÁLISIS ===")
        print(f"✓ Elementos clave cubiertos: {elementos_presentes}/{len(elementos_clave)}")
        print(f"✓ Análisis completo: {'Sí' if elementos_presentes >= 6 else 'Parcial'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar análisis complejo
cot_analisis_complejo()

=== CoT PARA ANÁLISIS COMPLEJO ===


CASO DE NEGOCIO:
Una startup de software tiene las siguientes métricas:
    - 10,000 usuarios activos mensuales
    - Tasa de conversión a premium: 5%
    - Precio premium: 29€/mes
    - Costo de adquisición por usuario: 15€
    - Retención mensual: 85%
    - Costos operativos mensuales: 12,000€

La empresa está considerando reducir el precio a 19€/mes para aumentar la conversión a 8%. 
¿Es una buena decisión financiera?

ANÁLISIS PASO A PASO:
**Análisis del caso de negocio**

### 1. Calcula métricas del escenario actual

* **Usuarios activos mensuales**: 10,000
* **Tasa de conversión a premium**: 5% = 0,05
* **Número de usuarios premium**: 10,000 * 0,05 = 500 usuarios
* **Ingresos mensuales por premium**: 500 usuarios * 29€/mes = 14,500€
* **Costo de adquisición total**: 10,000 usuarios * 15€ = 150,000€
* **Costo de adquisición por usuario premium**: 150,000€ / 500 usuarios = 300€
* **Retención mensual**: 85% = 0,85
* **Usuarios premium retenidos**: 500 usuarios * 0,85 = 425 usuarios


## Técnicas Avanzadas de CoT

In [8]:
# Técnica 1: CoT con Auto-Verificación
def cot_con_verificacion():
    print("=== CoT CON AUTO-VERIFICACIÓN ===")
    
    problema = "Una piscina se llena con dos bombas. La bomba A la llena en 4 horas, la bomba B en 6 horas. Si funcionan juntas, ¿en cuánto tiempo llenan la piscina?"
    
    prompt_verificacion = f"""Resuelve este problema paso a paso y luego verifica tu respuesta:
    
{problema}
    
PASO 1 - RESOLUCIÓN:
Piensa paso a paso para encontrar la solución.
    
PASO 2 - VERIFICACIÓN:
Revisa tu cálculo usando un método diferente o verificando que los números tienen sentido.
    
PASO 3 - RESPUESTA FINAL:
Confirma tu respuesta final.
    
Proceso completo:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_verificacion)])
        print("PROBLEMA:")
        print(problema)
        print("\nSOLUCIÓN CON AUTO-VERIFICACIÓN:")
        print(response.content)
        
        # Verificar estructura
        contenido = response.content.lower()
        tiene_resolucion = 'paso 1' in contenido or 'resolución' in contenido
        tiene_verificacion = 'paso 2' in contenido or 'verificación' in contenido
        tiene_final = 'paso 3' in contenido or 'final' in contenido
        
        print("\n=== ESTRUCTURA ===")
        print(f"✓ Resolución: {'Sí' if tiene_resolucion else 'No'}")
        print(f"✓ Verificación: {'Sí' if tiene_verificacion else 'No'}")
        print(f"✓ Respuesta final: {'Sí' if tiene_final else 'No'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar CoT con verificación
cot_con_verificacion()

=== CoT CON AUTO-VERIFICACIÓN ===


PROBLEMA:
Una piscina se llena con dos bombas. La bomba A la llena en 4 horas, la bomba B en 6 horas. Si funcionan juntas, ¿en cuánto tiempo llenan la piscina?

SOLUCIÓN CON AUTO-VERIFICACIÓN:
**PASO 1 - RESOLUCIÓN**

Para resolver este problema, necesitamos encontrar la tasa a la que cada bomba llena la piscina y luego combinar esas tasas para encontrar la tasa total cuando funcionan juntas.

La bomba A llena la piscina en 4 horas, lo que significa que su tasa de llenado es 1/4 de la piscina por hora.

La bomba B llena la piscina en 6 horas, lo que significa que su tasa de llenado es 1/6 de la piscina por hora.

Cuando funcionan juntas, sus tasas de llenado se suman. Entonces, la tasa total de llenado es:

(1/4 + 1/6) de la piscina por hora

Para sumar estas fracciones, necesitamos encontrar un denominador común, que en este caso es 12. Así que convertimos cada fracción:

(1/4) = (3/12)
(1/6) = (2/12)

Ahora podemos sumar:

(3/12 + 2/12) = 5/12 de la piscina por hora

Esto significa q

In [9]:
# Técnica 2: CoT Multi-Perspectiva
def cot_multi_perspectiva():
    print("=== CoT MULTI-PERSPECTIVA ===")
    
    dilema = """Una empresa de delivery está considerando implementar un algoritmo de IA 
    para optimizar rutas que podría reducir costos en 20% pero eliminaría 100 empleos 
    de repartidores. ¿Debería implementarlo?"""
    
    prompt_multi = f"""Analiza este dilema desde múltiples perspectivas:
    
{dilema}
    
Analiza paso a paso desde cada perspectiva:
    
PERSPECTIVA 1 - FINANCIERA:
- Beneficios económicos
- Costos de implementación
- ROI a corto y largo plazo
    
PERSPECTIVA 2 - SOCIAL/ÉTICA:
- Impacto en empleados
- Responsabilidad social corporativa
- Percepción pública
    
PERSPECTIVA 3 - ESTRATÉGICA:
- Competitividad del mercado
- Innovación tecnológica
- Sostenibilidad del negocio
    
SÍNTESIS:
- Integra las perspectivas
- Propone soluciones alternativas
- Recomienda curso de acción
    
Análisis completo:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_multi)])
        print("DILEMA EMPRESARIAL:")
        print(dilema)
        print("\nANÁLISIS MULTI-PERSPECTIVA:")
        print(response.content)
        
        # Verificar cobertura de perspectivas
        analisis = response.content.lower()
        perspectivas = ['financiera', 'social', 'ética', 'estratégica', 'síntesis']
        perspectivas_cubiertas = sum(1 for p in perspectivas if p in analisis)
        
        print(f"\n=== COBERTURA DEL ANÁLISIS ===")
        print(f"✓ Perspectivas cubiertas: {perspectivas_cubiertas}/{len(perspectivas)}")
        print(f"✓ Análisis integral: {'Sí' if perspectivas_cubiertas >= 4 else 'Parcial'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar análisis multi-perspectiva
cot_multi_perspectiva()

=== CoT MULTI-PERSPECTIVA ===


DILEMA EMPRESARIAL:
Una empresa de delivery está considerando implementar un algoritmo de IA 
    para optimizar rutas que podría reducir costos en 20% pero eliminaría 100 empleos 
    de repartidores. ¿Debería implementarlo?

ANÁLISIS MULTI-PERSPECTIVA:
**Análisis Completo del Dilema**

### PERSPECTIVA 1 - FINANCIERA

La perspectiva financiera se centra en los beneficios económicos y los costos asociados con la implementación del algoritmo de IA para optimizar rutas.

- **Beneficios económicos**: La reducción de costos en un 20% es un beneficio significativo. Esto podría traducirse en una mayor rentabilidad para la empresa, permitiéndole invertir en otras áreas o aumentar sus márgenes de beneficio.
- **Costos de implementación**: La implementación de un algoritmo de IA requiere una inversión inicial en tecnología, capacitación del personal y posibles ajustes en la infraestructura. Estos costos deben ser evaluados cuidadosamente para determinar si el ahorro a largo plazo justifica la i

## CoT para Debugging de Código

Una aplicación práctica muy útil: usar CoT para analizar y debuggear código.

In [10]:
# CoT para debugging de código
def cot_debugging():
    print("=== CoT PARA DEBUGGING DE CÓDIGO ===")
    
    codigo_con_bug = '''def calcular_promedio(numeros):
    total = 0
    for numero in numeros:
        total += numero
    promedio = total / len(numeros)
    return promedio

# Uso
datos = []
resultado = calcular_promedio(datos)
print(f"El promedio es: {resultado}")'''
    
    prompt_debugging = f"""Analiza este código paso a paso para encontrar problemas:
    
```python
{codigo_con_bug}
```
    
PASO 1 - COMPRENSIÓN:
¿Qué se supone que hace este código?
    
PASO 2 - ANÁLISIS LÍNEA POR LÍNEA:
Examina cada línea en busca de problemas potenciales.
    
PASO 3 - IDENTIFICACIÓN DE PROBLEMAS:
¿Qué errores o problemas específicos encuentras?
    
PASO 4 - CASOS PROBLEMÁTICOS:
¿En qué situaciones fallaría este código?
    
PASO 5 - SOLUCIÓN:
¿Cómo arreglarías estos problemas?
    
Análisis de debugging:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_debugging)])
        print("CÓDIGO A ANALIZAR:")
        print(codigo_con_bug)
        print("\nANÁLISIS DE DEBUGGING:")
        print(response.content)
        
        # Verificar si encontró el problema principal
        analisis = response.content.lower()
        encontro_division_cero = any(term in analisis for term in ['división', 'cero', 'vacía', 'empty'])
        propuso_solucion = 'if' in analisis or 'len(' in analisis or 'excepción' in analisis
        
        print(f"\n=== EFECTIVIDAD DEL DEBUGGING ===")
        print(f"✓ Identificó división por cero: {'Sí' if encontro_division_cero else 'No'}")
        print(f"✓ Propuso solución: {'Sí' if propuso_solucion else 'No'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar debugging
cot_debugging()

=== CoT PARA DEBUGGING DE CÓDIGO ===


CÓDIGO A ANALIZAR:
def calcular_promedio(numeros):
    total = 0
    for numero in numeros:
        total += numero
    promedio = total / len(numeros)
    return promedio

# Uso
datos = []
resultado = calcular_promedio(datos)
print(f"El promedio es: {resultado}")

ANÁLISIS DE DEBUGGING:
**PASO 1 - COMPRENSIÓN:**
Este código se supone que calcula el promedio de una lista de números. La función `calcular_promedio` toma una lista de números como entrada, suma todos los números y luego divide la suma por la cantidad de números para obtener el promedio. El resultado se imprime en la consola.

**PASO 2 - ANÁLISIS LÍNEA POR LÍNEA:**

1. `def calcular_promedio(numeros):` - Define una función llamada `calcular_promedio` que toma una lista de números como parámetro.
2. `total = 0` - Inicializa una variable `total` para almacenar la suma de los números.
3. `for numero in numeros:` - Itera sobre cada número en la lista `numeros`.
4. `total += numero` - Suma cada número a la variable `total`.
5. `

# Ejercicio con Chain of Thought

In [11]:
# Ejercicio final: Diseña tu propio CoT prompt
def ejercicio_cot():
    print("=== EJERCICIO: DISEÑA TU CoT PROMPT ===")
    print("\nTarea: Crear un sistema CoT para análisis de inversión")
    print("\nEscenario:")
    print("Una persona tiene 10,000€ para invertir y está considerando tres opciones:")
    print("1. Acciones de tech (retorno esperado 12% anual, riesgo alto)")
    print("2. Bonos gubernamentales (retorno 3% anual, riesgo bajo)")
    print("3. Fondo mixto (retorno 7% anual, riesgo medio)")
    print("\nLa persona es joven (25 años) y puede asumir riesgo moderado.")
    
    # Template para el estudiante
    template_cot = """
    # TU PROMPT CoT AQUÍ:
    
    Analiza esta decisión de inversión paso a paso:
    
    [ESCENARIO]
    
    PASO 1 - PERFIL DEL INVERSOR:
    - Analiza edad, tolerancia al riesgo, horizonte temporal
    
    PASO 2 - ANÁLISIS DE OPCIONES:
    - Evalúa cada opción: retorno, riesgo, liquidez
    
    PASO 3 - ESTRATEGIA DE DIVERSIFICACIÓN:
    - Considera combinar opciones
    
    PASO 4 - RECOMENDACIÓN:
    - Proporciona recomendación específica con justificación
    
    Análisis de inversión:
    """
    
    print("\nDiseña un prompt CoT estructurado:")
    print(template_cot)
    
    # Prompt de ejemplo bien diseñado
    prompt_ejemplo = """Analiza esta decisión de inversión usando razonamiento paso a paso:
    
SITUACIÓN:
Inversor de 25 años con 10,000€ considerando:
- Acciones tech: 12% retorno anual, riesgo alto
- Bonos: 3% retorno anual, riesgo bajo  
- Fondo mixto: 7% retorno anual, riesgo medio
Tolerancia: riesgo moderado
    
PASO 1 - PERFIL DEL INVERSOR:
Analiza edad, horizonte temporal y tolerancia al riesgo.
    
PASO 2 - EVALUACIÓN DE OPCIONES:
Para cada opción, calcula:
- Valor esperado en 10 años
- Nivel de riesgo vs. perfil
- Pros y contras específicos
    
PASO 3 - ESTRATEGIA DE PORTFOLIO:
Considera distribución óptima entre opciones basada en:
- Diversificación de riesgo
- Maximización de retorno ajustado por riesgo
- Liquidez y flexibilidad
    
PASO 4 - RECOMENDACIÓN FINAL:
Proporciona distribución específica (porcentajes) con:
- Justificación detallada
- Proyección a 10 años
- Consideraciones adicionales
    
Análisis completo:"""
    
    print("\n=== PROMPT DE REFERENCIA ===")
    print(prompt_ejemplo)
    
    # Ejecutar el prompt ejemplo
    print("\n=== RESULTADO DEL ANÁLISIS ===")
    try:
        response = llm.invoke([HumanMessage(content=prompt_ejemplo)])
        print(response.content)
        
        # Análisis de la respuesta
        analisis = response.content.lower()
        elementos = ['perfil', 'evaluación', 'portfolio', 'recomendación', '%', '€', 'años']
        elementos_presentes = sum(1 for elem in elementos if elem in analisis)
        
        print(f"\n=== CALIDAD DEL ANÁLISIS ===")
        print(f"✓ Elementos clave: {elementos_presentes}/{len(elementos)}")
        print(f"✓ Análisis completo: {'Sí' if elementos_presentes >= 5 else 'Parcial'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar ejercicio
ejercicio_cot()

=== EJERCICIO: DISEÑA TU CoT PROMPT ===

Tarea: Crear un sistema CoT para análisis de inversión

Escenario:
Una persona tiene 10,000€ para invertir y está considerando tres opciones:
1. Acciones de tech (retorno esperado 12% anual, riesgo alto)
2. Bonos gubernamentales (retorno 3% anual, riesgo bajo)
3. Fondo mixto (retorno 7% anual, riesgo medio)

La persona es joven (25 años) y puede asumir riesgo moderado.

Diseña un prompt CoT estructurado:

    # TU PROMPT CoT AQUÍ:

    Analiza esta decisión de inversión paso a paso:

    [ESCENARIO]

    PASO 1 - PERFIL DEL INVERSOR:
    - Analiza edad, tolerancia al riesgo, horizonte temporal

    PASO 2 - ANÁLISIS DE OPCIONES:
    - Evalúa cada opción: retorno, riesgo, liquidez

    PASO 3 - ESTRATEGIA DE DIVERSIFICACIÓN:
    - Considera combinar opciones

    PASO 4 - RECOMENDACIÓN:
    - Proporciona recomendación específica con justificación

    Análisis de inversión:
    

=== PROMPT DE REFERENCIA ===
Analiza esta decisión de inversión usa

**PASO 1 - PERFIL DEL INVERSOR**

El inversor tiene 25 años, lo que sugiere un horizonte temporal largo para sus inversiones. Con 10,000€ disponibles para invertir y una tolerancia al riesgo moderada, este inversor puede considerar una variedad de opciones de inversión. La edad y el horizonte temporal largo permiten asumir cierto nivel de riesgo, ya que hay tiempo para recuperarse de posibles pérdidas. Sin embargo, la tolerancia al riesgo moderada indica que el inversor no busca asumir riesgos extremos.

**PASO 2 - EVALUACIÓN DE OPCIONES**

### Acciones Tech: 12% retorno anual, riesgo alto

- **Valor esperado en 10 años**: Suponiendo un crecimiento constante del 12% anual, el valor esperado en 10 años sería de aproximadamente 30,959€ (10,000 * (1 + 0.12)^10).
- **Nivel de riesgo vs. perfil**: El riesgo alto asociado con las acciones tech es más alto que la tolerancia al riesgo moderada del inversor.
- **Pros y contras**:
  - Pros: Alto potencial de crecimiento a largo plazo.
  - Contra

## Limitaciones y Consideraciones de CoT

### ✅ Cuándo Usar CoT:
- Problemas matemáticos complejos
- Razonamiento lógico multi-paso
- Análisis que requiere transparencia
- Cuando necesitas verificar el proceso
- Problemas donde el "por qué" es importante

### ⚠️ Limitaciones:
- **Más tokens**: Respuestas más largas = mayor costo
- **Tiempo**: Razonamiento paso a paso toma más tiempo
- **Complejidad innecesaria**: Para problemas simples puede ser excesivo
- **Razonamiento erróneo**: Puede mostrar lógica incorrecta convincente

### 🎯 Mejores Prácticas:
1. **Estructura clara**: Define pasos específicos
2. **Verificación**: Incluye auto-verificación cuando sea posible
3. **Ejemplos**: Usa few-shot para mostrar patrón deseado
4. **Temperatura baja**: Para razonamiento más consistente
5. **Validación externa**: Verifica respuestas críticas independientemente

## Conceptos Clave Aprendidos

1. **Chain-of-Thought** mejora dramáticamente la precisión en problemas complejos
2. **Zero-shot CoT** es tan simple como agregar "piensa paso a paso"
3. **Few-shot CoT** combina ejemplos con razonamiento para mejor control
4. **Estructura explícita** guía el razonamiento hacia análisis completos
5. **Auto-verificación** aumenta la confiabilidad de las respuestas

## Próximos Pasos

En el siguiente notebook exploraremos **Técnicas Avanzadas** como Tree of Thoughts (ToT), Self-Consistency, y otras metodologías cutting-edge que llevan el prompt engineering al siguiente nivel.

### Para Practicar:
1. Aplica CoT a problemas de tu dominio específico
2. Experimenta con diferentes niveles de estructura
3. Compara precisión con y sin CoT
4. Desarrolla patrones de verificación personalizados